[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C70_RAG_Production_Course/01_ingestion/01_ingestion.ipynb)

# 01 · 文档摄取与解析（结构 / 顺序 / 噪声 / 元数据 / 去重 / 契约）

目标：把「解析丢了东西」这件事从模糊的担忧变成**六个可以测量的量**。

本 notebook 你会亲手实现：
1. **三种解析方式的可答性对比** —— 拍平后为什么检索指标正常但答案不可恢复
2. **双栏版面的阅读顺序** —— 按 y 排序 vs 先分栏，量出实体-属性错配率
3. **OCR 噪声的传导** —— 字符错误率 → 词级破坏率的理论值与实测值，以及词法/语义两条不同的衰减曲线
4. **元数据过滤的两面** —— precision 的收益，和写窄一格时 recall 的断崖
5. **近重复怎么占满 top-k** —— 有效独立文档数 eff@k
6. **幂等摄取契约** —— 状态机 + tombstone
7. **摄取质量门禁六项**

> 心智模型：**解析是流水线里唯一一个「做错了无法在下游补救」的环节。**

## 0 · 环境与工具函数

复用模块 00 的玩具嵌入（中文取单字 + 二元组）。

In [ ]:
import os, re, json, math, hashlib, random
from collections import Counter, defaultdict

import numpy as np

DIM = 4096
RNG = np.random.default_rng(0)

def tokenize(text):
    text = text.lower()
    toks = re.findall(r'[a-z0-9]+', text)
    for run in re.findall(r'[\u4e00-\u9fff]+', text):
        toks += list(run)
        toks += [run[i:i + 2] for i in range(len(run) - 1)]
    return toks

def embed(text, dim=DIM):
    v = np.zeros(dim)
    for tok in tokenize(text):
        v[int(hashlib.md5(tok.encode()).hexdigest(), 16) % dim] += 1.0
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def rank(cands, query, k=3):
    """cands: [(id, text)]，返回 [(id, text, score)] 按分数降序取 k。"""
    if not cands:
        return []
    mat = np.stack([embed(t) for _, t in cands])
    s = mat @ embed(query)
    order = np.argsort(-s)[:k]
    return [(cands[i][0], cands[i][1], float(s[i])) for i in order]

print('numpy', np.__version__, '| 工具就位')

## 1 · 结构损失：三种解析，同一份表格

关键观察：**拍平之后，检索照样召回那个块，块里每个字都在，但答案不可恢复。**

In [ ]:
TABLE_ROWS = [('北京', '2023', '1240'), ('北京', '2024', '1310'),
              ('上海', '2023', '1580'), ('上海', '2024', '1620')]

# 两个要恢复的关系：值与行名/年份的对应，以及值与列名的对应。
# 第二个常常被忘掉——但「1580 是营收还是成本」正是它决定的。
GOLD_ROW  = ('上海', '2023', '1580')     # 这个数属于哪一行
GOLD_COL  = ('上海', '营收', '1580')     # 这个数属于哪一列

def parse_structured(rows):
    """① 行转句：每行一句，行名、列名、值全部带上。"""
    return [f'{city} 在 {year} 年的营收是 {val} 万元。' for city, year, val in rows]

def parse_row_major(rows):
    """② 行主序拍平：按行把单元格连起来（很多解析器的默认行为）。"""
    cells = ['地区', '年份', '营收']
    for r in rows:
        cells += list(r)
    return [' '.join(cells)]

def parse_col_major(rows):
    """③ 列主序拍平：按列读——PDF 里按 x 坐标优先排序时的真实结果。"""
    return [' '.join(['地区'] + [r[0] for r in rows]
                     + ['年份'] + [r[1] for r in rows]
                     + ['营收'] + [r[2] for r in rows])]

def parse_row_major_chunked(rows, size=43):
    """④ 行主序拍平之后按字符数硬切。
    size=43 刻意让块边界落在「上海 2023 | 1580」这一行内部——
    在几百行的大表上，这件事必然会发生在某些行上。"""
    t = parse_row_major(rows)[0]
    return [t[i:i + size] for i in range(0, len(t), size)]

def locally_recoverable(pieces, rel, window=20):
    """判据：存在一个不超过 window 字符的局部窗口，同时含有关系的三个元素。

    为什么用「局部窗口」而不是「三个元素都在文本里」——
    后者对所有四种解析都成立（字一个都没少），完全区分不出可答与不可答。
    局部性才是模型能可靠利用的东西：跨越几十个 token 去重建表格的行列对应，
    正是模型最不可靠的操作之一。"""
    return any(all(x in t[i:i + window] for x in rel)
               for t in pieces for i in range(len(t)))

QUERY = '上海 2023 年的营收是多少'
print(f"{'解析方式':<16}{'片段':>5}{'top1 相似度':>13}{'字都在':>8}"
      f"{'行对应':>8}{'列对应':>8}")
res = {}
for name, fn in [('① 行转句', parse_structured),
                 ('② 行主序拍平', parse_row_major),
                 ('③ 列主序拍平', parse_col_major),
                 ('④ 行主序+硬切', parse_row_major_chunked)]:
    pieces = fn(TABLE_ROWS)
    top = rank([(f'{name}-{i}', t) for i, t in enumerate(pieces)], QUERY, k=1)[0]
    present = all(any(x in t for t in pieces) for x in set(GOLD_ROW + GOLD_COL))
    r_row = locally_recoverable(pieces, GOLD_ROW)
    r_col = locally_recoverable(pieces, GOLD_COL)
    res[name] = (top[2], present, r_row, r_col)
    print(f'{name:<16}{len(pieces):>5}{top[2]:>13.3f}{str(present):>8}'
          f'{str(r_row):>8}{str(r_col):>8}')

assert all(r[1] for r in res.values()), '四种解析里字都没少——所以「字在不在」不是有用的判据'
assert res['① 行转句'][2] and res['① 行转句'][3], '行转句必须同时保留行与列的对应'
assert res['② 行主序拍平'][2] and not res['② 行主序拍平'][3], \
    '行主序：值与行名相邻（能扛住），但列名丢在开头'
assert not res['③ 列主序拍平'][2] and not res['③ 列主序拍平'][3], '列主序两者全丢'
assert not res['④ 行主序+硬切'][2] and not res['④ 行主序+硬切'][3], '硬切两者全丢'
assert res['③ 列主序拍平'][0] > 0.3, '不可恢复的那个块，相似度依然不低'

print('\n✅ 三个结论，第二个最容易被想错：')
print('   ① 只有行转句同时保住了「哪一行」和「哪一列」。')
print('   ② 行主序拍平在小表上其实扛住了行对应——单元格与行名恰好相邻。')
print('      所以「拍平必然丢结构」是个错的说法。但它丢了**列名**：')
print('      「营收」只在开头出现一次，离 1580 有几十个 token。')
print('   ③ 真正让行对应也崩掉的是两个具体条件：列主序读取，以及块边界落在行内。')
print('      这两个条件在真实文档上都很常见——PDF 按 x 排序就得到列主序，')
print('      几百行的大表必然有若干行被边界切开。')
print('\n   注意最后一列以外的那一列：四种解析的「字都在」全是 True，')
print('   而不可恢复的块与查询的相似度依然 > 0.3。precision@1 满分，端到端错。')
print('\n   工程含义不是「拍平有时可以」，而是相反：不要把它当概率事件去赌。')
print('   行转句对 ③ 和 ④ 都免疫，实现只有五行。')

## 2 · 阅读顺序：双栏版面

合成一个双栏版面：左栏讲 A 产品，右栏讲 B 产品，每栏三段。
两种拼接方式，量出**实体-属性错配率**：一个块里同时出现两个产品名就算错配。

In [ ]:
# (x, y, text)：左栏 x=50，右栏 x=320
LAYOUT = [
    (50, 100, 'A 型机的额定功率是 1200 瓦。'),
    (50, 200, 'A 型机的工作温度范围是 0 到 40 摄氏度。'),
    (50, 300, 'A 型机的保修期是 24 个月。'),
    (320, 100, 'B 型机的额定功率是 2400 瓦。'),
    (320, 200, 'B 型机的工作温度范围是 -10 到 55 摄氏度。'),
    (320, 300, 'B 型机的保修期是 36 个月。'),
]

def order_by_y(blocks):
    """❌ 只按 y 排（单栏文档上是对的，双栏上交错）。"""
    return [t for _, _, t in sorted(blocks, key=lambda b: (b[1], b[0]))]

def order_by_column(blocks, gap=100):
    """✅ 先按 x 聚成栏，栏内按 y 排。"""
    xs = sorted({b[0] for b in blocks})
    cols, cur = [], [xs[0]]
    for x in xs[1:]:
        (cur.append(x) if x - cur[-1] < gap else (cols.append(cur), cur := [x]))
    cols.append(cur)
    out = []
    for col in cols:
        out += [t for _, _, t in sorted([b for b in blocks if b[0] in col],
                                        key=lambda b: b[1])]
    return out

def chunk_text(pieces, size=40):
    joined = ''.join(pieces)
    return [joined[i:i + size] for i in range(0, len(joined), size)]

def mismatch_rate(chunks, entities=('A 型机', 'B 型机')):
    """一个块里同时出现两个实体 → 实体-属性可能错配。"""
    bad = sum(1 for c in chunks if all(e in c for e in entities))
    return bad / len(chunks)

print(f"{'块大小':>7}{'按 y 排序':>12}{'先分栏':>10}")
bad_rates, good_rates = [], []
for size in [30, 40, 50, 60, 80]:
    mr_bad = mismatch_rate(chunk_text(order_by_y(LAYOUT), size))
    mr_good = mismatch_rate(chunk_text(order_by_column(LAYOUT), size))
    bad_rates.append(mr_bad); good_rates.append(mr_good)
    print(f'{size:>7}{mr_bad:>12.0%}{mr_good:>10.0%}')

print('\n首块对比:')
print('  按 y 排序:', chunk_text(order_by_y(LAYOUT), 40)[0])
print('  先分栏  :', chunk_text(order_by_column(LAYOUT), 40)[0])

assert all(g <= b for g, b in zip(good_rates, bad_rates)), '分栏在任何块大小下都不该更差'
assert np.mean(good_rates) < np.mean(bad_rates), '平均错配率必须下降'
assert min(good_rates) == 0.0, '至少在某些块大小下能做到零错配'
print(f'\n✅ 平均错配率从 {np.mean(bad_rates):.0%} 降到 {np.mean(good_rates):.0%}。')
print('   两个细节值得注意：')
print('   ① 分栏之后错配率不总是 0——**两栏交界处的那个块**天然会同时含两个实体。')
print('      这已经不是阅读顺序问题，而是块边界问题（模块 02 处理）。')
print('   ② 按 y 排序在单栏文档上完全正确，所以这个 bug 在测试文档上常常不出现；')
print('      它需要一份真正的双栏 PDF 才会暴露。')
print('   ③ 两种拼接产生的文本都「读起来通顺」——错误不会报错。')

## 3 · OCR 噪声：字符错误率怎么放大

两件事：
1. 验证 $P(\text{词被毁}) = 1-(1-\varepsilon)^L$——长词先坏。
2. 对比两条衰减曲线：**精确匹配**（词法）是断崖式的，**字符 n-gram 相似度**（语义替身）是渐进的。

In [ ]:
CONFUSE_POOL = '口日曰目臼白甲由申田巳己已'

def corrupt(text, eps, rng):
    """以 eps 的概率把每个字符替换成一个**不同的**字符（模拟 OCR 混淆）。

    注意「不同的」这三个字：如果替换池里包含原字符，
    实际错误率会低于 eps（本例低 1/13），理论公式就对不上了。
    这是一个真实存在的实现坑——注入故障时要确认故障真的注入了。"""
    out = []
    for ch in text:
        if rng.random() < eps:
            alt = [c for c in CONFUSE_POOL if c != ch]
            out.append(alt[int(rng.integers(len(alt)))])
        else:
            out.append(ch)
    return ''.join(out)

# --- 3a. 词级破坏率：理论 vs 实测 ---
print('词级破坏率（eps=3%）')
print(f"{'词长 L':>7}{'理论 1-(1-e)^L':>16}{'实测':>9}")
eps = 0.03
for L in [2, 4, 6, 8, 12]:
    word = '甲' * L
    rng = np.random.default_rng(L)
    trials = 20000
    broken = sum(1 for _ in range(trials) if corrupt(word, eps, rng) != word)
    theory = 1 - (1 - eps) ** L
    print(f'{L:>7}{theory:>16.3f}{broken / trials:>9.3f}')
    assert abs(theory - broken / trials) < 0.02, f'L={L} 理论与实测应当吻合'

print('\n✅ 长词先坏：L 从 2 到 12，破坏率从 %.1f%% 涨到 %.1f%%。'
      % (100 * (1 - 0.97 ** 2), 100 * (1 - 0.97 ** 12)))
print('   而专有名词、型号、法条编号恰恰都是长串。')

In [ ]:
# --- 3b. 两条衰减曲线 ---
DOC = '产品型号 XJ7720B 的额定功率是 1200 瓦，保修期为 24 个月。'
Q_TERMS = ['XJ7720B', '额定功率', '保修期']

def lexical_hit(noisy, terms):
    """词法检索的替身：所有关键词都必须精确出现。"""
    return all(t in noisy for t in terms)

def semantic_sim(noisy, clean):
    """语义检索的替身：字符 n-gram 相似度（降级是渐进的）。"""
    return float(np.dot(embed(noisy), embed(clean)))

print(f"{'eps':>6}{'词法精确命中率':>16}{'语义相似度':>13}")
lex_curve, sem_curve = [], []
for eps in [0.0, 0.005, 0.01, 0.03, 0.05, 0.10]:
    hits, sims = [], []
    for t in range(400):
        rng = np.random.default_rng(1000 * t + int(eps * 10000))
        noisy = corrupt(DOC, eps, rng)
        hits.append(lexical_hit(noisy, Q_TERMS))
        sims.append(semantic_sim(noisy, DOC))
    lex_curve.append(np.mean(hits)); sem_curve.append(np.mean(sims))
    print(f'{eps:>6.3f}{np.mean(hits):>16.2%}{np.mean(sims):>13.3f}')

# 词法的相对跌幅必须显著大于语义的
lex_drop = (lex_curve[0] - lex_curve[3]) / lex_curve[0]        # eps=3%
sem_drop = (sem_curve[0] - sem_curve[3]) / sem_curve[0]
print(f'\neps=3% 时：词法相对跌 {lex_drop:.0%}，语义相对跌 {sem_drop:.0%}')
assert lex_drop > 3 * sem_drop, '词法应当比语义衰减快得多'   # 实测约 5 倍
assert lex_curve[-1] < 0.3, 'eps=10% 时词法基本失效'
print(f'✅ 同一个噪声水平，词法路径的相对跌幅是语义路径的 {lex_drop / sem_drop:.1f} 倍。')
print('   这是「混合检索」在扫描件语料上的一个非常实际的动机（C11 模块 01）。')

## 4 · 元数据过滤：收益与断崖

加上正确的章节过滤 → precision 上升。
把过滤条件写窄一格 → **recall 直接掉到 0**，而检索层不会报错。

In [ ]:
CORPUS = [
    # (chunk_id, section, text)
    ('c1', '年假',   '正式员工每年享有 15 天带薪年假。'),
    ('c2', '年假',   '试用期员工每年享有 5 天带薪年假。'),
    ('c3', '病假',   '病假每年累计不超过 15 天，需提供医疗证明。'),
    ('c4', '婚假',   '婚假为 10 天，需在登记后一年内使用。'),
    ('c5', '报销',   '餐饮报销的单次上限是 200 元。'),
    ('c6', '报销',   '差旅报销的单次上限是 3000 元。'),
    ('c7', '考勤',   '每月迟到累计超过 3 次计一次警告。'),
    ('c8', '培训',   '每位员工每年有 2000 元培训预算。'),
]
Q = '正式员工的年假有多少天'
RELEVANT = {'c1'}

def search(corpus, query, k=3, sections=None):
    cands = [(cid, txt) for cid, sec, txt in corpus
             if sections is None or sec in sections]
    return [cid for cid, _, _ in rank(cands, query, k=k)], len(cands)

for label, secs in [('不过滤', None), ('正确过滤（年假）', {'年假'}),
                    ('写窄一格（病假）', {'病假'})]:
    got, n_cand = search(CORPUS, Q, k=3, sections=secs)
    prec = len(set(got) & RELEVANT) / len(got)
    rec = len(set(got) & RELEVANT) / len(RELEVANT)
    print(f'{label:<18} 候选 {n_cand:>2} | top3={got} | P@3 {prec:.2f} | R {rec:.2f}')

p_no, _ = search(CORPUS, Q, 3, None)
p_ok, _ = search(CORPUS, Q, 3, {'年假'})
p_bad, n_bad = search(CORPUS, Q, 3, {'病假'})
assert len(set(p_ok) & RELEVANT) / len(p_ok) > len(set(p_no) & RELEVANT) / len(p_no), \
    '正确过滤应当提升 precision'
assert len(set(p_bad) & RELEVANT) == 0, '过滤写错时 recall 是 0，不是「差一点」'
print('\n✅ 过滤是硬约束：它没有「排名靠后」这个中间状态。')
print('   所以「过滤后候选数为 0 或过小」必须是一个报警事件——')
print(f'   本例里写窄一格后候选只剩 {n_bad} 条，这个数本身就是信号。')

## 5 · 近重复占满 top-k

四份近重复文档（同一份合同的四个版本）+ 若干独立文档。
量 **eff@k**：top-k 里有多少个「内容簇」。

In [ ]:
def shingles(text, n=4):
    t = re.sub(r'\s+', '', text)
    return {t[i:i + n] for i in range(max(1, len(t) - n + 1))}

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 0.0

def cluster(corpus, thr=0.7):
    """按 Jaccard 把近重复聚成簇（并查集的朴素版）。"""
    ids = [cid for cid, _ in corpus]
    sh = {cid: shingles(t) for cid, t in corpus}
    parent = {i: i for i in ids}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            if jaccard(sh[ids[i]], sh[ids[j]]) >= thr:
                parent[find(ids[i])] = find(ids[j])
    return {cid: find(cid) for cid in ids}

BASE_CLAUSE = '乙方应在合同签订后 30 日内完成交付，逾期按日收取千分之一的违约金。'
DUPES = [(f'contract-v{v}', BASE_CLAUSE + f'（版本 {v}）') for v in range(1, 5)]
OTHERS = [
    ('attach-1', '附件说明：交付验收由甲方指定的第三方机构完成。'),
    ('policy-1', '公司差旅报销的单次上限是 3000 元。'),
    ('policy-2', '设备损坏需在两个工作日内报修。'),
]
CORP2 = DUPES + OTHERS
CL = cluster(CORP2)
print('内容簇:', Counter(CL.values()).most_common())

QC = '交付逾期的违约金怎么算'
def eff_at_k(corpus, clusters, query, k):
    top = [cid for cid, _, _ in rank(corpus, query, k=k)]
    return len({clusters[c] for c in top}), top

for k in [2, 3, 5]:
    eff, top = eff_at_k(CORP2, CL, QC, k)
    print(f'k={k}: top-k={top} | eff@k={eff} | eff/k={eff / k:.2f}')

eff5, _ = eff_at_k(CORP2, CL, QC, 5)
assert eff5 < 5, '近重复必然压低 eff@k'
# 去重后（每簇只留一份）再看
seen, deduped = set(), []
for cid, t in CORP2:
    if CL[cid] not in seen:
        seen.add(CL[cid]); deduped.append((cid, t))
eff5_dd, top_dd = eff_at_k(deduped, CL, QC, min(5, len(deduped)))
print(f'\n去重后语料 {len(deduped)} 条 | eff@{min(5, len(deduped))}={eff5_dd}')
assert eff5_dd > eff5, '摄取时去重应当提升有效独立信息量'
print(f'✅ eff@5 从 {eff5} 提升到 {eff5_dd}——同样的上下文预算买到更多信息。')
print('   注意结论不是「删掉重复文档」：合同版本要保留，')
print('   正确做法是保留最新版 + supersedes 链接，或在检索时按 doc_id 限流。')

## 6 · 幂等摄取契约

四条契约的实现。注意 `ingest_batch` 的返回值——
**它必须报告写了几个块、删了几个块，否则调用方无法断言任何东西。**

In [ ]:
class Store:
    def __init__(self):
        self.blocks = {}                 # (doc_id, version, idx) -> chunk
        self.doc_version = {}            # doc_id -> version
        self.doc_sha = {}                # doc_id -> content_sha
        self.tombstones = {}             # doc_id -> reason

    # --- 查询 ---
    def chunks_of(self, doc_id):
        return [k for k in self.blocks if k[0] == doc_id]

    def live_chunks(self):
        return [self.blocks[k] for k in self.blocks]

def split(text, size=40):
    return [text[i:i + size] for i in range(0, len(text), size)]

def ingest_batch(store, docs, size=40):
    """docs: [(doc_id, text)]。返回统计字典。"""
    st = dict(written=0, deleted=0, skipped=0, rejected=0)
    for doc_id, text in docs:
        if doc_id in store.tombstones:
            st['rejected'] += 1                     # 契约 4：拒绝被删过的文档
            continue
        sha = hashlib.sha256(text.encode()).hexdigest()[:12]
        if store.doc_sha.get(doc_id) == sha:
            st['skipped'] += 1                      # 契约 2：内容没变，整份跳过
            continue
        for k in store.chunks_of(doc_id):           # 契约 2：先删旧版本
            del store.blocks[k]; st['deleted'] += 1
        v = store.doc_version.get(doc_id, 0) + 1
        for i, piece in enumerate(split(text, size)):
            store.blocks[(doc_id, v, i)] = dict(    # 契约 3：主键 upsert
                doc_id=doc_id, version=v, chunk_index=i, text=piece,
                start=i * size, end=i * size + len(piece))
            st['written'] += 1
        store.doc_version[doc_id] = v; store.doc_sha[doc_id] = sha
    return st

def delete_doc(store, doc_id, reason='user_request'):
    n = 0
    for k in store.chunks_of(doc_id):
        del store.blocks[k]; n += 1
    store.tombstones[doc_id] = reason
    store.doc_sha.pop(doc_id, None)
    return n

# --- 走一遍状态机 ---
S = Store()
DOCS = [('d1', 'A' * 100), ('d2', 'B' * 60)]
print('首次摄取     ', ingest_batch(S, DOCS))
print('原样重放     ', ingest_batch(S, DOCS), '← 幂等：written 必须是 0')
print('d1 内容变了  ', ingest_batch(S, [('d1', 'A' * 100 + '新增一段')]))
n_del = delete_doc(S, 'd2')
print(f'删除 d2      删掉 {n_del} 个块')
print('全量同步重来 ', ingest_batch(S, DOCS), '← tombstone 拒绝了 d2')

versions = sorted({k[1] for k in S.chunks_of('d1')})
print(f'\nd1 当前在索引里的版本: {versions}')
replay = ingest_batch(S, DOCS)
assert replay['written'] == 0, '幂等性：重放不应写入任何块'
assert len(versions) == 1, f'同一时刻索引里只能有一个版本，实际 {versions}'
assert S.chunks_of('d2') == [], '删除必须在索引里生效'
assert replay['rejected'] == 1, 'tombstone 必须阻止被删文档被重新灌回'
print('\n✅ 四条契约都可验证。第 6 项门禁就是「把同一批数据再摄取一遍，断言 written == 0」。')
print(f'   一个值得注意的细节：d1 现在是 v{versions[0]} 而不是 v2——')
print('   因为「全量同步重来」那一步送进来的是**原始内容**，与 v2 不同，')
print('   于是它被正确地当成了又一次内容变更。')
print('   **版本号计的是变更次数，不是内容的身份**；内容回退不会复用旧版本号。')

## 7 · 摄取质量门禁：六项检查

确定性检查（2/3/6）阻断，统计检查（1/4/5）报警——与 C68 模块 04 的分级一致。

In [ ]:
REQUIRED_META = ('doc_id', 'version', 'source_uri', 'section_path')

def ingest_gate(report, baseline):
    """返回 (blocking, warnings)：两个 list。"""
    blocking, warn = [], []
    # 确定性 —— 零误报，直接阻断
    if report['n_tables'] > 0 and report['n_table_rows_rendered'] == 0:
        blocking.append('有表格但行转句为 0（表格被拍平）')
    missing = [k for k in REQUIRED_META if report['meta_complete'].get(k, 0) < 1.0]
    if missing:
        blocking.append(f'元数据不完整: {missing}')
    if report['replay_written'] != 0:
        blocking.append(f"摄取不幂等: 重放写入了 {report['replay_written']} 个块")
    # 统计 —— 只报警，阈值从基线推
    if report['blank_page_ratio'] > 0.05:
        warn.append(f"零文本页 {report['blank_page_ratio']:.1%} > 5%（可能没走 OCR）")
    lo = baseline['ocr_conf_mean'] - 2 * baseline['ocr_conf_sd']
    if report['ocr_conf'] < lo:
        warn.append(f"OCR 置信度 {report['ocr_conf']:.3f} < 基线-2σ ({lo:.3f})")
    if report['eff_at_10'] / 10 < 0.6:
        warn.append(f"eff@10/10 = {report['eff_at_10'] / 10:.2f} < 0.6（近重复过多）")
    return blocking, warn

BASELINE = dict(ocr_conf_mean=0.95, ocr_conf_sd=0.01)
HEALTHY = dict(n_tables=3, n_table_rows_rendered=42,
               meta_complete={k: 1.0 for k in REQUIRED_META},
               replay_written=0, blank_page_ratio=0.01, ocr_conf=0.96, eff_at_10=8)

cases = {
    '健康':        HEALTHY,
    '表格被拍平':  {**HEALTHY, 'n_table_rows_rendered': 0},
    '缺元数据':    {**HEALTHY, 'meta_complete': {**HEALTHY['meta_complete'], 'section_path': 0.4}},
    '不幂等':      {**HEALTHY, 'replay_written': 17},
    '扫描件没OCR': {**HEALTHY, 'blank_page_ratio': 0.31},
    '近重复过多':  {**HEALTHY, 'eff_at_10': 4},
}
for name, rep in cases.items():
    b, w = ingest_gate(rep, BASELINE)
    verdict = '阻断' if b else ('警告' if w else '通过')
    print(f'{name:<12} {verdict:<4} {(b + w)[0] if (b + w) else ""}')

assert ingest_gate(HEALTHY, BASELINE) == ([], [])
assert ingest_gate(cases['表格被拍平'], BASELINE)[0], '表格被拍平必须阻断'
assert ingest_gate(cases['不幂等'], BASELINE)[0], '不幂等必须阻断'
assert not ingest_gate(cases['近重复过多'], BASELINE)[0], '统计类只报警不阻断'
assert ingest_gate(cases['近重复过多'], BASELINE)[1], '但必须报警'
print('\n✅ 六项门禁。分级的理由：确定性检查误报率为零，可以阻断；')
print('   统计检查会误报，阻断它们是「门禁被关掉」这一结局的起点（C68-04）。')

## ✏️ 练习 1：阅读顺序恢复（带页眉页脚）

在第 2 节的分栏基础上加两件真实的事：
- **页眉/页脚**：y 极小或极大的窄块，必须剔除
- **跨栏标题**：宽度接近页宽的块，必须排在所有栏之前

实现 `reading_order(blocks, page_w=400)`，`blocks` 是 `(x, y, w, text)`。
返回正确顺序的 text 列表。

In [ ]:
def reading_order(blocks, page_w=400, header_y=60, footer_y=740, col_gap=100):
    """返回按正确阅读顺序排列的 text 列表。
    规则：① 丢掉 y < header_y 或 y > footer_y 的块；
          ② w >= 0.8 * page_w 的块是跨栏标题，按 y 排在最前；
          ③ 其余按 x 聚栏（间隔 >= col_gap 算新栏），栏内按 y 排。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
PAGE = [
    (50,  20, 300, '第 3 页  内部资料'),          # 页眉 → 丢
    (50,  70, 340, '第二章 产品参数'),            # 跨栏标题（w=340 >= 320）
    (50, 120, 150, 'A 型机的额定功率是 1200 瓦。'),
    (50, 220, 150, 'A 型机的保修期是 24 个月。'),
    (320, 120, 150, 'B 型机的额定功率是 2400 瓦。'),
    (320, 220, 150, 'B 型机的保修期是 36 个月。'),
    (50, 780, 300, '- 12 -'),                     # 页脚 → 丢
]
got = reading_order(PAGE)
assert len(got) == 5, f'页眉页脚应当被剔除，得到 {len(got)} 段'
assert got[0] == '第二章 产品参数', '跨栏标题排最前'
assert got[1].startswith('A 型机的额定功率'), '左栏在前'
assert got[2].startswith('A 型机的保修期'), '栏内按 y'
assert got[3].startswith('B 型机的额定功率'), '再到右栏'
assert '内部资料' not in ''.join(got) and '- 12 -' not in ''.join(got)
print('✅ 练习 1 通过：标题 → 左栏 → 右栏，页眉页脚已剔除')

## 📖 参考答案 1

In [ ]:
# 练习 1 参考答案
def reading_order(blocks, page_w=400, header_y=60, footer_y=740, col_gap=100):
    body = [b for b in blocks if header_y <= b[1] <= footer_y]
    spans = [b for b in body if b[2] >= 0.8 * page_w]
    rest = [b for b in body if b[2] < 0.8 * page_w]
    out = [t for _, _, _, t in sorted(spans, key=lambda b: b[1])]
    xs = sorted({b[0] for b in rest})
    cols, cur = [], []
    for x in xs:
        if cur and x - cur[-1] >= col_gap:
            cols.append(cur); cur = []
        cur.append(x)
    if cur:
        cols.append(cur)
    for col in cols:
        out += [t for _, _, _, t in
                sorted([b for b in rest if b[0] in col], key=lambda b: b[1])]
    return out

got = reading_order(PAGE)
assert len(got) == 5 and got[0] == '第二章 产品参数'
assert got[1].startswith('A 型机的额定功率') and got[3].startswith('B 型机的额定功率')
assert '内部资料' not in ''.join(got) and '- 12 -' not in ''.join(got)
print('✅ 参考答案 1 通过')
print('   三条规则的顺序不能换：先剔页眉页脚，再抽跨栏标题，最后才分栏。')
print('   反过来做的话，页眉（也是宽块）会被当成跨栏标题排到最前面。')

## ✏️ 练习 2：OCR 噪声下的关键词存活率

实现 `survival_rate(terms, eps)`：解析式地算出「查询里所有关键词都精确存活」的概率
$\prod_i (1-\varepsilon)^{L_i}$，并实现 `survival_empirical` 用采样验证。

然后回答一个工程问题：**给定 eps 与关键词，最多能容忍几个关键词？**
实现 `max_terms_under(eps, target, word_len)` —— 在存活率不低于 `target` 的前提下，
查询最多能含几个长度为 `word_len` 的关键词。

In [ ]:
def survival_rate(terms, eps):
    """所有 term 都精确存活的理论概率。"""
    # TODO
    raise NotImplementedError

def max_terms_under(eps, target, word_len):
    """返回满足 (1-eps)^(m*word_len) >= target 的最大整数 m（m >= 0）。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
assert abs(survival_rate(['abcd'], 0.0) - 1.0) < 1e-12
assert abs(survival_rate(['ab'], 0.1) - 0.81) < 1e-9
assert abs(survival_rate(['ab', 'cdef'], 0.1) - 0.9 ** 6) < 1e-9

# 采样验证
def survival_empirical(terms, eps, trials=4000, seed=0):
    rng = np.random.default_rng(seed)
    ok = 0
    for _ in range(trials):
        ok += all(corrupt(t, eps, rng) == t for t in terms)
    return ok / trials
th = survival_rate(['甲乙丙', '丁戊'], 0.05)
em = survival_empirical(['甲乙丙', '丁戊'], 0.05)
print(f'理论 {th:.3f} vs 实测 {em:.3f}')
assert abs(th - em) < 0.03, '理论与实测应当吻合'

assert max_terms_under(0.03, 0.8, 6) == 1, max_terms_under(0.03, 0.8, 6)
assert max_terms_under(0.01, 0.8, 6) == 3, max_terms_under(0.01, 0.8, 6)
assert max_terms_under(0.0, 0.8, 6) >= 100, 'eps=0 时不受限'
print('✅ 练习 2 通过：eps=3% 时，6 字关键词最多只能要求 1 个精确命中')

## 📖 参考答案 2

In [ ]:
# 练习 2 参考答案
def survival_rate(terms, eps):
    total_len = sum(len(t) for t in terms)
    return (1 - eps) ** total_len

def max_terms_under(eps, target, word_len):
    if eps <= 0:
        return 10 ** 6
    m = 0
    while (1 - eps) ** ((m + 1) * word_len) >= target:
        m += 1
        if m > 10 ** 6:
            break
    return m

assert abs(survival_rate(['ab'], 0.1) - 0.81) < 1e-9
assert abs(survival_rate(['ab', 'cdef'], 0.1) - 0.9 ** 6) < 1e-9
assert max_terms_under(0.03, 0.8, 6) == 1
assert max_terms_under(0.01, 0.8, 6) == 3
assert max_terms_under(0.0, 0.8, 6) >= 100
print('✅ 参考答案 2 通过')
print('   这个函数的用途是定容量：它告诉你在给定 OCR 质量下，')
print('   「要求 N 个关键词同时精确命中」这个检索策略还成不成立。')
print('   eps=3% 时答案是 1——也就是说纯词法检索已经不能作为唯一召回路径。')

## ✏️ 练习 3：块级去重 + 保留最新版

实现 `dedup_keep_latest(corpus, thr=0.7)`：
`corpus` 是 `[(doc_id, version, text)]`。把近重复聚簇，**每簇只保留 version 最大的那条**，
并返回 `(kept, supersedes)`，其中 `supersedes[被丢弃的 doc_id] = 保留的 doc_id`。

这是第 5 节说的「不要直接删」的落地：被丢弃的版本要能被追溯。

In [ ]:
def dedup_keep_latest(corpus, thr=0.7):
    """返回 (kept, supersedes)。
    kept: [(doc_id, version, text)]，每个内容簇一条（version 最大）
    supersedes: {丢弃的 doc_id: 保留的 doc_id}"""
    # TODO：复用第 5 节的 shingles / jaccard
    raise NotImplementedError

In [ ]:
# —— 自测 ——
C3 = [('k-v1', 1, BASE_CLAUSE + '（版本 1）'),
      ('k-v3', 3, BASE_CLAUSE + '（版本 3）'),
      ('k-v2', 2, BASE_CLAUSE + '（版本 2）'),
      ('other', 1, '公司差旅报销的单次上限是 3000 元。'),
      ('other2', 1, '设备损坏需在两个工作日内报修。')]
kept, sup = dedup_keep_latest(C3, thr=0.7)
kept_ids = sorted(d for d, _, _ in kept)
assert kept_ids == ['k-v3', 'other', 'other2'], kept_ids
assert sup == {'k-v1': 'k-v3', 'k-v2': 'k-v3'}, sup
# 每条被丢弃的都能追溯到保留的那条
assert all(v in kept_ids for v in sup.values())
# 不该跨内容合并
assert 'other' not in sup and 'other2' not in sup
print(f'✅ 练习 3 通过：{len(C3)} 条 → 保留 {len(kept)} 条，{len(sup)} 条可追溯')

## 📖 参考答案 3

In [ ]:
# 练习 3 参考答案
def dedup_keep_latest(corpus, thr=0.7):
    ids = [c[0] for c in corpus]
    meta = {c[0]: (c[1], c[2]) for c in corpus}
    sh = {c[0]: shingles(c[2]) for c in corpus}
    parent = {i: i for i in ids}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            if jaccard(sh[ids[i]], sh[ids[j]]) >= thr:
                parent[find(ids[i])] = find(ids[j])
    groups = defaultdict(list)
    for cid in ids:
        groups[find(cid)].append(cid)
    kept, sup = [], {}
    for members in groups.values():
        winner = max(members, key=lambda c: (meta[c][0], c))
        kept.append((winner, meta[winner][0], meta[winner][1]))
        for m in members:
            if m != winner:
                sup[m] = winner
    return kept, sup

kept, sup = dedup_keep_latest(C3, thr=0.7)
assert sorted(d for d, _, _ in kept) == ['k-v3', 'other', 'other2']
assert sup == {'k-v1': 'k-v3', 'k-v2': 'k-v3'}
print('✅ 参考答案 3 通过')
print('   supersedes 链是关键：用户问「旧版本里写的是什么」时你还答得上来，')
print('   而如果直接删掉，这个问题就永久失去了答案。')

## ✏️ 练习 4：完整的摄取报告生成器

把前面所有量凑成一份报告，喂给第 7 节的 `ingest_gate`。

实现 `make_report(store, docs, tables, metas, ocr_conf, blank_pages, n_pages)`：
- `replay_written`：原样重放一次 `ingest_batch` 得到的 `written`
- `meta_complete`：每个必需字段的完整率（有值且非空的比例）
- `eff_at_10`：对一个固定探针查询算 eff@10（语料不足 10 条时按实际条数算簇数）
- 其余直接透传

In [ ]:
def make_report(store, docs, tables, metas, ocr_conf, blank_pages, n_pages,
                probe='交付逾期的违约金怎么算'):
    """返回可直接喂给 ingest_gate 的 dict。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
S2 = Store()
D2 = [('a', BASE_CLAUSE + '（版本 1）'), ('b', BASE_CLAUSE + '（版本 2）'),
      ('c', '公司差旅报销的单次上限是 3000 元。')]
ingest_batch(S2, D2, size=200)
METAS = [dict(doc_id='a', version=1, source_uri='s3://a', section_path='第一章'),
         dict(doc_id='b', version=1, source_uri='s3://b', section_path='第一章'),
         dict(doc_id='c', version=1, source_uri='s3://c', section_path='')]   # 缺一个
rep = make_report(S2, D2, tables=2, metas=METAS,
                  ocr_conf=0.96, blank_pages=1, n_pages=50)
print({k: v for k, v in rep.items() if k != 'meta_complete'})
print('meta_complete:', rep['meta_complete'])

assert rep['replay_written'] == 0, '重放必须为 0'
assert abs(rep['blank_page_ratio'] - 0.02) < 1e-9
assert abs(rep['meta_complete']['section_path'] - 2 / 3) < 1e-9
assert rep['meta_complete']['doc_id'] == 1.0
b, w = ingest_gate(rep, BASELINE)
assert any('元数据' in x for x in b), '缺 section_path 必须阻断'
print(f'✅ 练习 4 通过：门禁给出 {len(b)} 项阻断、{len(w)} 项警告')

## 📖 参考答案 4

In [ ]:
# 练习 4 参考答案
def make_report(store, docs, tables, metas, ocr_conf, blank_pages, n_pages,
                probe='交付逾期的违约金怎么算'):
    replay = ingest_batch(store, docs, size=200)
    complete = {}
    for key in REQUIRED_META:
        got = sum(1 for m in metas if m.get(key) not in (None, '', []))
        complete[key] = got / len(metas) if metas else 0.0
    corpus = [(c['doc_id'] + f"#{c['chunk_index']}", c['text'])
              for c in store.live_chunks()]
    k = min(10, len(corpus))
    if corpus:
        cl = cluster(corpus, thr=0.7)
        top = [cid for cid, _, _ in rank(corpus, probe, k=k)]
        eff = len({cl[c] for c in top})
    else:
        eff = 0
    return dict(n_tables=tables,
                n_table_rows_rendered=tables * 14 if tables else 0,
                meta_complete=complete,
                replay_written=replay['written'],
                blank_page_ratio=blank_pages / n_pages,
                ocr_conf=ocr_conf,
                eff_at_10=eff)

rep = make_report(S2, D2, tables=2, metas=METAS,
                  ocr_conf=0.96, blank_pages=1, n_pages=50)
assert rep['replay_written'] == 0
assert abs(rep['blank_page_ratio'] - 0.02) < 1e-9
assert abs(rep['meta_complete']['section_path'] - 2 / 3) < 1e-9
b, w = ingest_gate(rep, BASELINE)
assert any('元数据' in x for x in b)
print('✅ 参考答案 4 通过')
print('   报告生成器里唯一有技巧的一行是 replay——')
print('   它不检查代码，而是把同一批数据再摄取一遍并断言什么都没发生。')
print('   这是幂等性唯一可靠的验证方式（C68-00 的 run ∘ run = run）。')

## 🧪 真实工程胶囊：接真实解析库

```python
# ══════════════════════════════════════════════════════════════════
# A. 解析：拿三样东西，不是一样（讲解第 1 节）
# ══════════════════════════════════════════════════════════════════
from unstructured.partition.auto import partition

def parse(path):
    els = partition(filename=path, strategy='hi_res',      # hi_res 才有坐标
                    infer_table_structure=True)             # ← 表格必须开
    out = []
    section_stack = []
    for e in els:
        kind = type(e).__name__            # Title / NarrativeText / Table / ListItem
        if kind == 'Title':
            depth = getattr(e.metadata, 'category_depth', 0) or 0
            section_stack = section_stack[:depth] + [e.text]
        if kind == 'Table':
            # 表格走单独路径：行转句 + 保留 markdown 原文
            html = e.metadata.text_as_html
            for sentence in table_rows_to_sentences(html):
                out.append(dict(text=sentence, element_type='table',
                                table_html=html,
                                section_path=' / '.join(section_stack),
                                page=e.metadata.page_number))
            continue
        out.append(dict(text=e.text, element_type=kind,
                        section_path=' / '.join(section_stack),
                        page=e.metadata.page_number,
                        coords=e.metadata.coordinates))    # ← 阅读顺序要用
    return out

# ══════════════════════════════════════════════════════════════════
# B. 阅读顺序：只在需要时接管（讲解第 3 节）
# ══════════════════════════════════════════════════════════════════
#   多数解析库已经做了排序；但双栏 PDF 上要自己验证。
#   便宜的做法：抽 5 页人工看一遍，把结论写进摄取文档。
#   自动化的做法：x 坐标直方图双峰检测 → 命中就用 XY-cut 重排。

# ══════════════════════════════════════════════════════════════════
# C. OCR：记录置信度，不要顺便用 LLM 清洗（讲解第 4 节）
# ══════════════════════════════════════════════════════════════════
import pytesseract
data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)
page_conf = np.mean([c for c in data['conf'] if c > 0]) / 100.0
meta['ocr_conf'] = page_conf
if page_conf < OCR_REVIEW_THRESHOLD:
    review_queue.put(path)                # 人工复核，而不是自动改写

# ══════════════════════════════════════════════════════════════════
# D. 摄取：幂等 + tombstone（讲解第 7 节）
# ══════════════════════════════════════════════════════════════════
def ingest(path, col):
    sha = sha256_file(path)
    if tombstoned(path):        return 'rejected'
    if current_sha(path) == sha: return 'skipped'          # 幂等
    v = bump_version(path)
    col.delete(where={'doc_id': path, 'version': {'$lt': v}})   # 先删旧
    chunks = chunk(parse(path))                                  # 再写新
    col.upsert(ids=[f'{path}:{v}:{i}' for i in range(len(chunks))], ...)
    return 'written'

def delete(path, col, reason):
    n = col.count(where={'doc_id': path})
    col.delete(where={'doc_id': path})
    write_tombstone(path, reason)
    assert col.count(where={'doc_id': path}) == 0, '删除必须可验证'
    return n

# ══════════════════════════════════════════════════════════════════
# E. CI（讲解第 8 节）
# ══════════════════════════════════════════════════════════════════
# 1) 对一批固定样本文档跑摄取 → 生成 report
# 2) ingest_gate(report, baseline)：确定性项阻断，统计项报警
# 3) 幂等自检：同一批重放，断言 written == 0
# 4) 基线（ocr_conf 的均值与方差）从最近 30 次摄取滚动计算（C68-04）
```

---

## 小结

| 结论 | 数字 / 判据 | 在哪一节 |
|---|---|---|
| 解析是唯一「下游无法补救」的环节 | 结构、顺序、位置丢了就是丢了 | 讲解 1 |
| 拍平不是必然丢结构；列主序与块边界才是 | 行主序小表可恢复；③④ 不可恢复但相似度 > 0.3 | 第 1 节 |
| 按 y 排序在双栏上产生实体错配 | 平均错配率 77% → 17%（交界处的块无法为 0） | 第 2 节 |
| OCR 噪声优先毁掉长词 | $1-(1-\varepsilon)^L$；ε=3% 时 12 字词 31% 被毁 | 第 3 节 |
| 词法检索衰减比语义快约 5 倍 | ε=3% 时词法相对跌 35%、语义 7% | 第 3 节 |
| 元数据过滤没有「排名靠后」这个中间态 | 写窄一格 → recall 直接 0 | 第 4 节 |
| 近重复占满 top-k，浪费上下文预算 | eff@5 从 2 提升到 4 | 第 5 节 |
| 幂等性只能靠「重放并断言无事发生」验证 | `replay_written == 0` | 第 6/7 节 |

下一模块：**02 · 分块**——把「块该多大」从口味问题变成一个有约束的优化问题。